### Ejercicio: Datos climáticos estación

Link de datos: https://drive.google.com/drive/folders/1QD_GVY_KDRu6CEtN3qUmybuatw0WjRYx?usp=sharing


Crea un nuevo DataFrame con datos de 1 estaciones SENAMHI de diferentes regiones del Perú (https://www.senamhi.gob.pe/site/descarga-datos/).

Considerar lo siguiente a partir de "tutorial-para-la-descarga-de-datos.pdf".

Los datos de las columnas corresponden a:
- Columna A: Año
- Columna B: Mes
- Columna C: Día
- Columna D: Precipitación acumulada
- Columna E: Temperatura máxima
- Columna F: Temperatura mínima

En las celdas donde aparece -99.9 significa que no hay información disponible para esa variable

Luego:
1. Parsear el nombres de la estación Chosica y leer el archivo.
1. Calcula el promedio mensual de precipitación
1. Calcula el promedio anual de precipitación
2. Identifica el día más lluvioso
4. Exporta los resultados formateados en CSV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Tu código aquí – Ejercicio: Estación

# Leer los datos de las estaciones y crear un DataFrame combinado
import os
files = os.listdir('/content/Senamhi')  # Lista los archivos en el directorio
files

['qc00151205_CANCHACALLA.txt',
 'qc00000543_ñaña.txt',
 'tutorial-para-la-descarga-de-datos.pdf',
 'qc00151209_chosica.txt',
 'qc00155224_santiago_de_tuna.txt',
 'qc00155213_santa_eulalia.txt']

In [ ]:
# Parsear el nombre del archivo para obtener el código y nombre de la estación
station = files[5]
codigo = station.split("_")
print(codigo)
# nombre =

#print(f"Código: {codigo}, Nombre: {nombre}")

['qc00155213', 'santa', 'eulalia.txt']


In [11]:
import pandas as pd
station_df = pd.read_table(f"/content/Senamhi/{station}", sep = r"\s+", header = None, na_values=["-99.9", "-999", "-3256"] ) # Sin encabezados
station_df.columns = ['Año', 'Mes', 'Dia', 'Pr','Tmax', 'Tmin']  # Asignar nombres de columnas

station_df.dtypes # cuenta cuántos NaN hay por columna
# Formatear las columnas
# station_df['Año'] = station_df['Año'].astype(int)
#

,0
Año,int64
Mes,int64
Dia,int64
Pr,float64
Tmax,float64
Tmin,float64


In [22]:
# Precipitacion mensual
precipitacion_mensual = station_df.groupby("Mes")["Pr"].mean()
precipitacion_mensual
# Precipitacion total anual
precipitacion_anual = station_df.groupby("Año")[ "Pr"].sum()
precipitacion_anual
# Dia mas lluvioso
dia_mas_lluvioso = station_df.loc[station_df["Pr"].idxmax()]
dia_mas_lluvioso
print(f"Precipitación mensual:\n{precipitacion_mensual}\n")
print(f"Precipitación anual:\n{precipitacion_anual}\n")

print(f"El día más lluvioso fue el {dia_mas_lluvioso['Año']:.0f}-{dia_mas_lluvioso['Mes']:.0f}-{dia_mas_lluvioso['Dia']:.0f} con {dia_mas_lluvioso['Pr']} mm de precipitación.")

Precipitación mensual:
Mes
1     0.319776
2     0.543553
3     0.525434
4     0.022222
5     0.006583
6     0.000142
7     0.000000
8     0.001317
9     0.010278
10    0.023051
11    0.013830
12    0.109941
Name: Pr, dtype: float64

Precipitación anual:
Año
1963      0.5
1964     27.5
1965      6.6
1966     97.9
1967    205.5
1968      1.1
1969     56.3
1970    144.0
1971     63.9
1972    162.7
1973    144.9
1974     72.3
1975     56.6
1976    120.1
1977     28.5
1978     28.7
1979     23.0
1980     23.0
1981     12.8
1982     26.9
1983     33.0
1984     19.4
1985      0.6
1986      6.3
1987      0.9
1988     39.7
1989     62.7
1990     19.1
1991     10.8
1992      0.5
1993      5.6
1994     32.2
1995     14.6
1996     31.8
1997     22.8
1998     59.3
1999     70.2
2000     35.3
2001     39.4
2002     39.3
2003     14.4
2004     18.2
2005      6.5
2006     44.8
2007     20.0
2008     48.7
2009     65.2
2010     19.3
2011     32.5
2012     38.8
2013     21.4
2014      7.2
Name: Pr, dtyp

### Ejercicio 2: Datos climáticos multiestación

Crea un nuevo DataFrame con datos de 5 estaciones SENAMHI de diferentes regiones del Perú (https://www.senamhi.gob.pe/site/descarga-datos/).

Luego:
1. Paresear el nombres de la estaciones y leer los archivo.
1. Unir las db (Fecha, Est1, Est2, ...)
1. Calcula el promedio mensual de precipitación por región
2. Identifica el mes más lluvioso en cada estación
4. Exporta el resultado formateado a CSV

In [26]:
import os
import pandas as pd
from functools import reduce

path = '/content/Senamhi'
lista_archivos = os.listdir(path)

est = {}
for x in lista_archivos:
    if x.endswith('.txt'):
        c = x.split('_',1)[0]
        n = x.split('_',1)[1]
        n = n.replace('.txt','')
        data = pd.read_table(path+'/'+x, sep=r'\s+', header=None, na_values=['-99.9','-999','-3256'])
        data.columns=['Año','Mes','Dia','Pr','Tmax','Tmin']
        est[n]=data

print(est.keys())

lista2 = []
for k in est:
    d = est[k]
    d['Fecha']=pd.to_datetime(dict(year=d.Año,month=d.Mes,day=d.Dia))
    aux = d[['Fecha','Pr']]
    aux = aux.rename(columns={'Pr':k})
    lista2.append(aux)

df_final = lista2[0]
for i in range(1,len(lista2)):
    df_final = pd.merge(df_final, lista2[i], on='Fecha', how='outer')

df_final=df_final.sort_values('Fecha')
df_final = df_final.reset_index(drop=True)
print(df_final.head())

df_final['Mes']=df_final['Fecha'].dt.month
cols_est = list(est.keys())
prom_mensual = df_final.groupby('Mes')[cols_est].mean()
print(prom_mensual)

mes_max = prom_mensual.idxmax()
print(mes_max)

prom_mensual.to_csv('/content/precipitacion_mensual_multiestacion.csv')
mes_max.to_csv('/content/mes_mas_lluvioso.csv',header=['Mes_mas_lluvioso'])

dict_keys(['CANCHACALLA', 'ñaña', 'chosica', 'santiago_de_tuna', 'santa_eulalia'])
       Fecha  CANCHACALLA  ñaña  chosica  santiago_de_tuna  santa_eulalia
0 1963-12-01          NaN   NaN      NaN               NaN            0.0
1 1963-12-02          NaN   NaN      NaN               NaN            0.0
2 1963-12-03          NaN   NaN      NaN               NaN            0.0
3 1963-12-04          NaN   NaN      NaN               NaN            0.0
4 1963-12-05          NaN   NaN      NaN               NaN            0.2
     CANCHACALLA      ñaña   chosica  santiago_de_tuna  santa_eulalia
Mes                                                                  
1       1.849940  0.026185  0.178629          1.784516       0.319776
2       3.080680  0.033954  0.307066          3.439153       0.543553
3       2.781514  0.020926  0.154278          3.186570       0.525434
4       0.555641  0.001027  0.084370          0.567551       0.022222
5       0.021712  0.000701  0.006732          0.04167